# divide and conquer
1. 문제를 더 작은 문제로 나누어서 해결하는 접근법
2. 문제를 나눈 후에 여러개의 cpu에서 동시에 처리되도록 함으로써 계산 시간을 단축시킬 수 있음

In [ ]:
import pandas as pd

In [ ]:
#대량의 데이터를 읽어올때는... usecols를 이용해서 필요한 컬럼만 읽어옵니다.
df = pd.read_csv('zinc_db/AAAA.txt', sep='\t', usecols=['smiles','logp'])

In [ ]:
df

In [ ]:
import glob

# 패턴에 매칭되는 파일 경로를 리스트로 가져옴. (저는 샘플로 100개만 다운 받아봤어요)
# 별표(*)는 모든 경우르 의미합니다. *.txt면 모든 .txt 파일을 선택합니다.
txt_files = glob.glob('./zinc_db/*.txt')

print(f"매칭된 파일 개수: {len(txt_files)}")
# 상위 5개 파일명만 출력해서 확인
print("파일 샘플:", txt_files[-5:])

In [ ]:
# 패턴에 매칭되는 파일 경로를 리스트로 가져오는데,
# 파일 이름이 A로 시작하는 파일만 선택합니다. (A*.txt 패턴을 보이는 파일만 선택)
# 현재 각 학생별로 다운 받은 파일 수가 다르기 때문에.. 일단 A로 시작하는 파일만 선택해서 실험해보겠습니다.

txt_files = glob.glob('./zinc_db/A*.txt')

print(f"매칭된 파일 개수: {len(txt_files)}")
# 상위 5개 파일명만 출력해서 확인
print("파일 샘플:", txt_files[-5:])

In [ ]:
# 각 txt 파일들에 들어있는 물질 개수를 확인해봤습니다. 파일마다 제각각입니다.
for f in txt_files:
    df = pd.read_csv(f, sep='\t', usecols=['smiles','logp'])
    print (df.shape)

In [ ]:
df

In [ ]:
#df의 logp컬럼에서 최소값을 확인하는 방법
logp_min = df['logp'].min()
logp_min

In [ ]:
#logp 값이 가장 작은 smiles 코드 찾는 방법
df_low = df.loc[df['logp']==logp_min,'smiles']
df_low

In [ ]:
type(df_low)

In [ ]:
# pandas Series에서 필요한 smiles 코드 정보만 추출하는 방법
df.loc[df['logp']==logp_min,'smiles'].item()

In [ ]:
# pandas Series에서 필요한 smiles 코드 정보만 리스트로 추출하는 방법
df.loc[df['logp']==logp_min,'smiles'].to_list()

In [ ]:
#리스트에 있는 문자를 하나의 str으로 묶는 방법
', '.join(['A','B','C'])

In [ ]:
#만약에 최소값이 smiles 코드가 여러개라면? list로 추출한 후에 join함수를 이용해서 하나의 str으로 묶을 수 있음.
', '.join(df.loc[df['logp']==logp_min,'smiles'].to_list())

# Quiz1
### 코드 흐름도
1. 가장 단순한 검색 방법
2. 각 파일에서 가장 작은 logp 값을 찾는다.
3. 그 다음 파일에서 더 작은 logp 값을 찾는다면, 더 작은 logp 값으로 새로 저장한다.
4. 모든 파일을 확인하면서 더 작은 logp 값이 있는지 확인해본다.
### 아래 셀에서 코드를 채워주세요
1. 주석에 해당하는 부분을 코드로 변경해주세요
2. 각 주석은 코드 한줄에 해당됩니다. 한줄 이상 적을 필요가 없으니, 각 줄의 빈칸을 채워주세요.

In [ ]:
import time

start_time = time.time()

lowest_logp = float('inf') #무한대를 의미. 초기 값을 가장 큰 값으로 선택.
lowest_smiles = '' #smiles코드는 그냥 빈 str
for f in txt_files:
    df = #pd.read_csv를 이용해서 각 파일 읽어오기
    logp_min = #df['logp']에서 최소값 구하기
    smi_min = #logp가 가장 낮은 smiles 코드 선택
    if logp_min < lowest_logp: #새로운 파일에서 찾아낸 logp 값이 기존에 lowest_logp에 저장된 logp보다 더 낮다면
        #lowest_logp에 logp_min 값을 저장. (새로운 값으로 업데이트)
        #lowest_smi에 smi_min 값을 저장. (새로운 값으로 업데이트)

end_time = time.time()
print ('smiles:', lowest_smi)
print ('logp:', lowest_logp)
print ('total computation time:', end_time - start_time)

In [ ]:
#https://www.dask.org/
#dask는 jupyter notebook 환경에서 padnas 연산을 병렬화 할 수 있는 라이브러리입니다.

import dask.dataframe as dd
from dask.distributed import Client

# 1. 로컬 클러스터 시작 (주피터에서 대시보드로 현황 모니터링 가능)
client = Client() 

'''
사용할 cpu 개수를 조절하려면 Client()안에 어떤 파라미터를 쓸 수 있을까요?
(LLM에게 물어보면 찾아줄거에요. 2개의 parameter가 있습니다)

- 힌트 -
1. core 개수
2. thread 개수
'''

client

In [ ]:
# 2. 2,000개 파일을 하나의 가상 데이터프레임으로 연결 (Lazy Loading)
# Lazy Loading은 코드 라인의 실행을 미루는 방식입니다. 한번에 계산하지 않고,
# 계산에 필요한 모든 정보가 모일 때 까지 기다렸다가 연산을 수행하는 방식입니다.
# 파일명이 'zinc_*.csv' 형태라면 와일드카드로 한 번에 로드

start_time = time.time()
df = dd.read_csv('./zinc_db/A*.txt', sep='\t', usecols=['smiles', 'logp'])
end_time = time.time()

# 전체 행 개수 확인 (실제 데이터를 스캔하므로 시간이 걸릴 수 있음)
row_count = len(df) 
print(f"Total rows: {row_count:,}")
print ("Time spent:", end_time - start_time)

In [ ]:
# 파티션 개수 확인 (확인해보니, 분석할 txt 파일 개수인 것 같네요)
print(f"Number of partitions: {df.npartitions}")

# 데이터의 스키마(컬럼명, 데이터 타입) 및 지연 상태 확인
print(df)

In [ ]:
# 데이터의 구조와 타입 확인
df.info()

In [ ]:
# 3. 최솟값 계산 (분할 정복 알고리즘이 내부적으로 실행됨)
# 실제 연산은 .compute()를 호출할 때까지 시작되지 않음

start_time = time.time()
min_logp = df['logp'].min().compute()
result = df[df['logp']==min_logp].compute()
end_time = time.time()

print(f"최저 logP 물질: {result}")
print ('time for computation:',end_time-start_time) #그런데... 분산 계산이 더 느립니다. 무엇이 문제일까요?
# 위 코드를 프롬프트로 해서, 아래 cell에서 compute() 실행 횟수를 1회로 바꿔봅시다.

In [ ]:
#Dask는 compute()가 실행된 순간 연산을 실행합니다.
#compute() 횟수를 줄여야 합니다. (1회 실행이면 충분)
#LLM의 도움을 받아서 compute()를 한번만 실행하는 코드로 바꿔 봅시다.
#compute() 실행 횟수가 줄었을 때, 연산 시간이 얼마나 줄어드는지 확인해봅시다.
#이제까지는 'A*.txt' 파일로 테스트 해봤습니다. 코드가 완성되면, 모든 .txt 파일에 대해서 계산이 얼마나 걸리는지 테스트 해봅시다.

In [ ]:
# 각 파티션별 메모리 사용량 확인 방법 (바이트 단위)
memory_usage = df.memory_usage(deep=True).compute()
print(f"Total Memory: {memory_usage.sum() / 1e9:.2f} GB")

In [ ]:
#계산이 끝난 후 client를 종료합니다.
#(client를 종료하지 않으면, 처음에 선언된 clinet가 port를 계속 점유하고 있음. 계산후 port를 비워주기 위함)
client.close()